In [ ]:
import os
import re
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

ROOT = Path.cwd()
if ROOT.name == "scripts":
    ROOT = ROOT.parent

# Unpacked Zenodo archive. Expected subdirectories:
#   workflows/  normalized_workflows/  54_workflows_to_label/
DATASET_DIR = Path("../zenodo_dataset")
if not DATASET_DIR.is_absolute():
    DATASET_DIR = (Path.cwd() / DATASET_DIR).resolve()

DATA_CSV = ROOT / "dataset" / "workflow_metadata.csv"
WORKFLOWS_DIR = DATASET_DIR / "workflows"
WF_DIR = WORKFLOWS_DIR
OUT_DIR = ROOT / "accuracy"
LABEL_DIR = OUT_DIR / "workflows_to_label"

N_SAMPLE = 54          # ~2% of the dataset
SEED     = 42          # reproducibility

OUT_DIR.mkdir(parents=True, exist_ok=True)
LABEL_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT      :", ROOT)
print("DATA_CSV  :", DATA_CSV, "->", DATA_CSV.exists())
print("WF_DIR    :", WF_DIR, "->", WF_DIR.exists())
print("OUT_DIR   :", OUT_DIR)


In [2]:
# --- Load workflows (content + identity) ------------------------------------
# Primary source: metadata CSV (has the full YAML in `content` plus repo/workflow_file).
# Fallback: read the flattened files in the unpacked Zenodo workflows/ directory.

def flatten_name(repo: str, workflow_file: str) -> str:
    """Reproduce the flattened filename used in the collected workflow dataset."""
    repo_clean = str(repo).replace("/", "_")
    wf_clean = str(workflow_file).lstrip("/").replace("/", "__").replace("\\", "__")
    return f"{repo_clean}__{wf_clean}"

if DATA_CSV.exists():
    meta = pd.read_csv(DATA_CSV)
    if "content" not in meta.columns:
        raise ValueError("workflow_metadata.csv has no `content` column; use the *_no-content? no.")
    meta = meta[["repo", "workflow_file", "content"]].dropna(subset=["content"]).copy()
else:
    # Fallback: reconstruct from flattened files (repo/workflow_file are approximate)
    rows = []
    for p in sorted(WF_DIR.glob("*.y*ml")):
        rows.append({"repo": p.stem, "workflow_file": p.name,
                     "content": p.read_text(encoding="utf-8", errors="replace")})
    meta = pd.DataFrame(rows)

# Stable identity + the flattened source filename we will copy from later.
meta["workflow_id"] = meta["repo"].astype(str) + "::" + meta["workflow_file"].astype(str)
meta = meta.drop_duplicates(subset=["workflow_id"]).reset_index(drop=True)
meta["src_filename"] = [flatten_name(r, w) for r, w in zip(meta["repo"], meta["workflow_file"])]

print(f"Loaded {len(meta)} unique workflows from {'CSV' if DATA_CSV.exists() else 'flattened files'}.")
meta.head(3)[["repo", "workflow_file", "src_filename"]]

Loaded 2722 unique workflows from CSV.


,repo,workflow_file,src_filename
0,Azure/AKS,.github/workflows/codespell.yml,Azure_AKS__.github__workflows__codespell.yml
1,Azure/AKS,.github/workflows/website-check.yml,Azure_AKS__.github__workflows__website-check.yml
2,Azure/AKS,.github/workflows/website-publish.yml,Azure_AKS__.github__workflows__website-publish...


In [3]:
# --- Constants: taint sources (ARGUS), well-known owners, purpose keywords ----

# Exact attacker-controllable context references (from ARGUS, see diversityMetrics.md 4.2)
TAINT_EXACT = [
    "github.event.issue.body", "github.event.pull_request.body",
    "github.event.comment.body", "github.event.review.body",
    "github.event.review_comment.body", "github.event.head_commit.message",
    "github.event.discussion.body", "github.event.discussion_comment.body",
    "github.event.discussion.title", "github.event.changes.body.from",
    "github.event.workflow_run.head_commit.message",
    "github.event.check_run.output.text", "github.event.check_run.output.summary",
    "github.event.check_suite.head_commit.message",
    "github.event.issue.title", "github.event.pull_request.title",
    "github.event.pull_request.head.ref", "github.event.pull_request.head.label",
    "github.event.pull_request.head.repo.default_branch",
    "github.event.pull_request.head.repo.description",
    "github.event.pull_request.head.repo.homepage",
    "github.event.changes.title.from",
    "github.event.workflow.path", "github.event.workflow.name",
    "github.event.workflow_run.path", "github.event.workflow_run.head_branch",
    "github.event.workflow_run.head_repository.description",
    "github.event.workflow_run.display_title",
    "github.event.check_run.output.title",
    "github.event.check_run.check_suite.head_branch",
    "github.event.check_suite.head_branch",
    "github.event.deployment.environment",
    "github.event.deployment.original_environment",
    "github.event.deployment_status.environment",
    "github.event.deployment_status.environment_url",
    "github.head_ref",
]

# Wildcard patterns (ARGUS `matches("...%...")` -> regex)
TAINT_PATTERNS = [
    r"github\.event\.commits[\w\[\]\.]*\.message",
    r"github\.event\.commits[\w\[\]\.]*\.author\.[\w\.]+",
    r"github\.event\.commits[\w\[\]\.]*\.committer\.[\w\.]+",
    r"github\.event\.client_payload[\w\[\]\.]*",
    r"github\.event\.head_commit\.author\.[\w\.]+",
    r"github\.event\.head_commit\.committer\.[\w\.]+",
    r"github\.event\.workflow_run\.head_commit\.author\.[\w\.]+",
    r"github\.event\.workflow_run\.head_commit\.committer\.[\w\.]+",
    r"github\.event\.check_run\.pull_requests[\w\[\]\.]*\.head\.ref",
    r"github\.event\.check_suite\.pull_requests[\w\[\]\.]*\.head\.ref",
    r"github\.event\.check_suite\.head_commit\.author\.[\w\.]+",
    r"github\.event\.check_suite\.head_commit\.committer\.[\w\.]+",
    r"github\.event\.pages[\w\[\]\.]*\.page_name",
    r"github\.event\.pages[\w\[\]\.]*\.title",
]
TAINT_PATTERNS_RE = [re.compile(p) for p in TAINT_PATTERNS]

WELL_KNOWN_OWNERS = {
    "actions", "github", "azure", "aws-actions", "docker", "google-github-actions",
    "hashicorp", "microsoft", "gradle", "codecov", "dependabot",
}

# High/medium/low trigger classification (diversityMetrics.md 2.3)
HIGH_TRIGGERS = {"pull_request_target", "issue_comment", "issues",
                 "discussion_comment", "discussion"}
MEDIUM_TRIGGERS = {"pull_request", "workflow_run"}

PURPOSE_KEYWORDS = [
    ("security-scan", ["codeql", "security", "scan", "sast", "snyk", "trivy", "semgrep"]),
    ("dependency-management", ["dependabot", "deps", "dependency", "renovate"]),
    ("code-quality", ["lint", "format", "style", "quality", "prettier", "eslint"]),
    ("deploy-release", ["deploy", "release", "publish"]),
    ("documentation", ["docs", "pages", "documentation"]),
    ("build", ["build", "compile", "package"]),
    ("ci-test", ["ci", "test", "check", "validate", "verify"]),
]

SHA_RE = re.compile(r"^[0-9a-f]{40}$")
EXPR_RE = re.compile(r"\$\{\{")

def normalize_on(parsed):
    """Return the value of the `on:` key (PyYAML may turn `on` into the bool True)."""
    if not isinstance(parsed, dict):
        return None
    if "on" in parsed:
        return parsed["on"]
    if True in parsed:
        return parsed[True]
    return None

def trigger_key_set(on_val):
    if on_val is None:
        return set()
    if isinstance(on_val, str):
        return {on_val}
    if isinstance(on_val, list):
        return {str(x) for x in on_val}
    if isinstance(on_val, dict):
        return {str(k) for k in on_val.keys()}
    return set()

print("Constants ready:", len(TAINT_EXACT), "exact taint refs,",
      len(TAINT_PATTERNS), "taint patterns.")

Constants ready: 36 exact taint refs, 14 taint patterns.


In [4]:
# --- Metric extraction: one workflow YAML -> full diversity feature dict ------

def _iter_steps(jobs):
    """Yield (job, step) for every step across all jobs."""
    if not isinstance(jobs, dict):
        return
    for job in jobs.values():
        if not isinstance(job, dict):
            continue
        steps = job.get("steps")
        if isinstance(steps, list):
            for step in steps:
                if isinstance(step, dict):
                    yield job, step

def _permission_scope(perm):
    if perm is None:
        return "not set"
    if isinstance(perm, str):
        return perm if perm in ("read-all", "write-all") else "granular"
    if isinstance(perm, dict):
        return "empty" if len(perm) == 0 else "granular"
    return "granular"

def _workflow_purpose(name, filename):
    text = f"{name or ''} {filename or ''}".lower()
    for label, kws in PURPOSE_KEYWORDS:
        if any(kw in text for kw in kws):
            return label
    return "other"

def extract_metrics(content: str, workflow_file: str = "") -> dict:
    text = content or ""
    text_clean = text.replace("\t", "    ")
    try:
        parsed = yaml.safe_load(text_clean)
    except Exception:
        parsed = None
    if not isinstance(parsed, dict):
        parsed = {}
    parse_ok = bool(parsed)

    jobs = parsed.get("jobs", {}) if isinstance(parsed.get("jobs"), dict) else {}

    # ---- Structural ----------------------------------------------------------
    num_jobs = len(jobs)
    steps = list(_iter_steps(jobs))
    num_steps = len(steps)
    has_matrix = any(isinstance(j, dict) and isinstance(j.get("strategy"), dict)
                     and "matrix" in j["strategy"] for j in jobs.values() if isinstance(j, dict))
    has_needs = any(isinstance(j, dict) and j.get("needs") for j in jobs.values() if isinstance(j, dict))
    loc = len(text.splitlines())

    # ---- Triggers ------------------------------------------------------------
    on_val = normalize_on(parsed)
    triggers = trigger_key_set(on_val)
    num_triggers = len(triggers)
    if triggers & HIGH_TRIGGERS:
        trigger_risk = "high"
    elif triggers & MEDIUM_TRIGGERS:
        trigger_risk = "medium"
    else:
        trigger_risk = "low"
    has_dispatch_inputs = bool(
        isinstance(on_val, dict) and isinstance(on_val.get("workflow_dispatch"), dict)
        and on_val["workflow_dispatch"].get("inputs")
    )

    # ---- Actions -------------------------------------------------------------
    uses_vals, run_count = [], 0
    uses_reusable = False
    for job in jobs.values():
        if isinstance(job, dict) and isinstance(job.get("uses"), str):
            uses_reusable = True  # job-level uses: -> reusable workflow
    for _job, step in steps:
        if isinstance(step.get("uses"), str):
            uses_vals.append(step["uses"].strip())
        if "run" in step:
            run_count += 1

    third_party, owners, pinned = [], set(), 0
    for u in uses_vals:
        if u.startswith("./"):
            continue  # local action
        if u.startswith("docker://"):
            third_party.append(u)
            continue
        if re.search(r"\.ya?ml(@|$)", u):
            uses_reusable = True
            continue  # reusable workflow reference
        third_party.append(u)
        owner = u.split("/")[0]
        owners.add(owner.lower())
        ref = u.split("@", 1)[1] if "@" in u else ""
        if SHA_RE.match(ref):
            pinned += 1

    num_third_party = len(third_party)
    num_owners = len(owners)
    sha_ratio = (pinned / num_third_party) if num_third_party else 0.0
    action_run_ratio = (len(uses_vals) / run_count) if run_count else float(len(uses_vals))
    uses_obscure = any(o not in WELL_KNOWN_OWNERS for o in owners)

    # ---- Security-relevant ---------------------------------------------------
    has_expr_in_run = False
    for _job, step in steps:
        run = step.get("run")
        if isinstance(run, str) and EXPR_RE.search(run):
            has_expr_in_run = True
            break

    taint_count = sum(text.count(t) for t in TAINT_EXACT)
    taint_count += sum(len(rx.findall(text)) for rx in TAINT_PATTERNS_RE)

    perm = parsed.get("permissions")
    has_permissions = "permissions" in parsed or any(
        isinstance(j, dict) and "permissions" in j for j in jobs.values() if isinstance(j, dict))
    permission_scope = _permission_scope(perm)

    secrets_count = len(re.findall(r"secrets\.", text))
    github_token = ("github.token" in text.replace(" ", "")) or ("secrets.GITHUB_TOKEN" in text)

    env_with_expr = False
    def _env_has_expr(obj):
        if isinstance(obj, dict) and isinstance(obj.get("env"), dict):
            for v in obj["env"].values():
                if isinstance(v, str) and EXPR_RE.search(v):
                    return True
        return False
    env_with_expr = _env_has_expr(parsed) or any(_env_has_expr(j) for j in jobs.values() if isinstance(j, dict)) \
        or any(_env_has_expr(s) for _j, s in steps)

    has_conditional = ("if" in parsed) or any(
        (isinstance(j, dict) and "if" in j) for j in jobs.values() if isinstance(j, dict)) \
        or any("if" in s for _j, s in steps)

    # ---- Ecosystem -----------------------------------------------------------
    runners = set()
    for job in jobs.values():
        if isinstance(job, dict):
            ro = job.get("runs-on")
            if isinstance(ro, str):
                runners.add(ro)
            elif isinstance(ro, list):
                runners.update(str(x) for x in ro)
            elif isinstance(ro, dict):
                lbl = ro.get("labels")
                if isinstance(lbl, str):
                    runners.add(lbl)
                elif isinstance(lbl, list):
                    runners.update(str(x) for x in lbl)
    has_self_hosted = any("self-hosted" in r.lower() for r in runners)
    uses_container = any(isinstance(j, dict) and (j.get("container") or j.get("services"))
                         for j in jobs.values() if isinstance(j, dict))

    name = parsed.get("name") if isinstance(parsed.get("name"), str) else ""
    purpose = _workflow_purpose(name, workflow_file)

    return {
        # structural
        "num_jobs": num_jobs, "num_steps": num_steps, "has_matrix": int(has_matrix),
        "has_needs": int(has_needs), "uses_reusable": int(uses_reusable), "loc": loc,
        # triggers
        "num_triggers": num_triggers, "trigger_risk": trigger_risk,
        "has_dispatch_inputs": int(has_dispatch_inputs),
        # actions
        "num_third_party": num_third_party, "num_owners": num_owners,
        "sha_ratio": round(sha_ratio, 4), "action_run_ratio": round(action_run_ratio, 4),
        "uses_obscure": int(uses_obscure),
        # security
        "has_expr_in_run": int(has_expr_in_run), "taint_count": taint_count,
        "has_permissions": int(has_permissions), "permission_scope": permission_scope,
        "secrets_count": secrets_count, "github_token": int(github_token),
        "env_with_expr": int(env_with_expr), "has_conditional": int(has_conditional),
        # ecosystem
        "has_self_hosted": int(has_self_hosted), "uses_container": int(uses_container),
        "workflow_purpose": purpose, "parse_ok": int(parse_ok),
    }

# quick smoke test on the first workflow
extract_metrics(meta.iloc[0]["content"], meta.iloc[0]["workflow_file"])

{'num_jobs': 1,
 'num_steps': 2,
 'has_matrix': 0,
 'has_needs': 0,
 'uses_reusable': 0,
 'loc': 19,
 'num_triggers': 2,
 'trigger_risk': 'medium',
 'has_dispatch_inputs': 0,
 'num_third_party': 2,
 'num_owners': 2,
 'sha_ratio': 0.0,
 'action_run_ratio': 2.0,
 'uses_obscure': 1,
 'has_expr_in_run': 0,
 'taint_count': 0,
 'has_permissions': 1,
 'permission_scope': 'granular',
 'secrets_count': 0,
 'github_token': 0,
 'env_with_expr': 0,
 'has_conditional': 0,
 'has_self_hosted': 0,
 'uses_container': 0,
 'workflow_purpose': 'other',
 'parse_ok': 1}

In [ ]:
# --- Compute metrics for every workflow --------------------------------------
records = []
for _, row in meta.iterrows():
    m = extract_metrics(row["content"], row["workflow_file"])
    m["workflow_id"] = row["workflow_id"]
    m["repo"] = row["repo"]
    m["workflow_file"] = row["workflow_file"]
    m["src_filename"] = row["src_filename"]
    records.append(m)

feat = pd.DataFrame.from_records(records)

# Derived primary stratification axes (diversityMetrics.md "Suggested Sampling Strategy")
def complexity_tier(r):
    if r["num_jobs"] >= 4 or r["num_steps"] >= 10 or r["has_matrix"]:
        return "complex"
    if r["num_jobs"] >= 2 or r["num_steps"] >= 4:
        return "medium"
    return "simple"

def action_profile(r):
    if r["num_third_party"] == 0:
        return "no-third-party"
    return "pinned" if r["sha_ratio"] >= 0.5 else "unpinned"

def security_surface(r):
    if r["taint_count"] > 0 and (r["has_expr_in_run"] or r["env_with_expr"]):
        return "taint"
    if r["has_expr_in_run"]:
        return "expr-no-taint"
    return "no-expr"

feat["complexity"]       = feat.apply(complexity_tier, axis=1)
feat["action_profile"]   = feat.apply(action_profile, axis=1)
feat["security_surface"] = feat.apply(security_surface, axis=1)

feat_path = OUT_DIR / "workflow_diversity_features.csv"
feat.to_csv(feat_path, index=False)
print(f"Computed {len(feat)} feature rows -> {feat_path}")
print(f"Parse failures: {(feat['parse_ok'] == 0).sum()}")
feat.head(3)

In [6]:
# --- Population distribution across the primary diversity axes ----------------
for col in ["complexity", "trigger_risk", "action_profile", "security_surface",
            "workflow_purpose", "permission_scope"]:
    dist = feat[col].value_counts()
    pct = (dist / len(feat) * 100).round(1)
    print(f"\n== {col} ==")
    print(pd.concat([dist.rename("n"), pct.rename("%")], axis=1).to_string())


== complexity ==
               n     %
complexity            
complex     1130  41.5
medium       868  31.9
simple       724  26.6

== trigger_risk ==
                 n     %
trigger_risk            
medium        1327  48.8
low           1081  39.7
high           314  11.5

== action_profile ==
                   n     %
action_profile            
unpinned        2048  75.2
pinned           510  18.7
no-third-party   164   6.0

== security_surface ==
                     n     %
security_surface            
no-expr           1711  62.9
expr-no-taint      847  31.1
taint              164   6.0

== workflow_purpose ==
                          n     %
workflow_purpose                 
other                  1097  40.3
ci-test                 661  24.3
deploy-release          357  13.1
build                   266   9.8
security-scan           133   4.9
code-quality            105   3.9
dependency-management    53   1.9
documentation            50   1.8

== permission_scope ==
        

In [7]:
# --- Build the standardized feature matrix used for diversity clustering ------
# Heavy-tailed counts -> log1p, then z-score. Booleans kept as 0/1. Categoricals one-hot.

COUNT_COLS = ["num_jobs", "num_steps", "loc", "num_triggers",
              "num_third_party", "num_owners", "taint_count", "secrets_count"]
RATIO_COLS = ["sha_ratio", "action_run_ratio"]
BOOL_COLS  = ["has_matrix", "has_needs", "uses_reusable", "has_dispatch_inputs",
              "uses_obscure", "has_expr_in_run", "has_permissions", "github_token",
              "env_with_expr", "has_conditional", "has_self_hosted", "uses_container"]
CAT_COLS   = ["trigger_risk", "action_profile", "complexity",
              "permission_scope", "workflow_purpose"]

def build_matrix(df: pd.DataFrame) -> np.ndarray:
    parts = []
    # counts -> log1p + z-score
    for c in COUNT_COLS:
        x = np.log1p(pd.to_numeric(df[c], errors="coerce").fillna(0).clip(lower=0).to_numpy())
        sd = x.std() or 1.0
        parts.append(((x - x.mean()) / sd).reshape(-1, 1))
    # ratios -> z-score (action_run_ratio can be large -> log1p first)
    for c in RATIO_COLS:
        raw = pd.to_numeric(df[c], errors="coerce").fillna(0).to_numpy()
        x = np.log1p(raw) if c == "action_run_ratio" else raw
        sd = x.std() or 1.0
        parts.append(((x - x.mean()) / sd).reshape(-1, 1))
    # booleans as-is
    for c in BOOL_COLS:
        parts.append(pd.to_numeric(df[c], errors="coerce").fillna(0).to_numpy().reshape(-1, 1))
    # one-hot categoricals
    dummies = pd.get_dummies(df[CAT_COLS].astype(str), prefix=CAT_COLS)
    parts.append(dummies.to_numpy(dtype=float))
    return np.hstack(parts)

X_all = build_matrix(feat)
print("Feature matrix:", X_all.shape)

Feature matrix: (2722, 44)


In [8]:
# --- Diversity selection primitives ------------------------------------------
# select_diverse(X, k): pick k maximally-representative REAL rows.
# Prefers k-medoids-style (KMeans centroids -> nearest unique real point);
# falls back to farthest-first (max-min) traversal if scikit-learn is absent.

try:
    from sklearn.cluster import KMeans
    _HAVE_SKLEARN = True
except Exception:
    _HAVE_SKLEARN = False

def _maxmin(X, k, seed):
    rng = np.random.default_rng(seed)
    n = len(X)
    if k >= n:
        return list(range(n))
    start = int(rng.integers(n))
    chosen = [start]
    d = np.linalg.norm(X - X[start], axis=1)
    while len(chosen) < k:
        nxt = int(np.argmax(d))
        chosen.append(nxt)
        d = np.minimum(d, np.linalg.norm(X - X[nxt], axis=1))
    return chosen

def select_diverse(X, k, seed=SEED):
    n = len(X)
    if k <= 0:
        return []
    if k >= n:
        return list(range(n))
    if not _HAVE_SKLEARN:
        return _maxmin(X, k, seed)
    km = KMeans(n_clusters=k, random_state=seed, n_init=10).fit(X)
    chosen, used = [], set()
    for c in range(k):
        centroid = km.cluster_centers_[c]
        order = np.argsort(np.linalg.norm(X - centroid, axis=1))
        for idx in order:
            if idx not in used:
                used.add(int(idx)); chosen.append(int(idx)); break
    return chosen

# --- Allocation across security-surface strata (sqrt-proportional) -----------
# sqrt-proportional over-weights the rarer, scanner-critical strata (taint / expr)
# so the sample has enough likely-positive cases to measure recall, while still
# reflecting the population. Guarantees >=1 per non-empty stratum.
STRATUM_COL = "security_surface"
STRATUM_ORDER = ["taint", "expr-no-taint", "no-expr"]

sizes = {s: int((feat[STRATUM_COL] == s).sum()) for s in STRATUM_ORDER}
sizes = {s: n for s, n in sizes.items() if n > 0}
weights = {s: np.sqrt(n) for s, n in sizes.items()}
wsum = sum(weights.values())
alloc = {s: max(1, int(round(N_SAMPLE * w / wsum))) for s, w in weights.items()}

# fix rounding so the allocation sums exactly to N_SAMPLE
def _fix_total(alloc, target, sizes):
    while sum(alloc.values()) != target:
        diff = target - sum(alloc.values())
        if diff > 0:  # add to the stratum with most remaining headroom
            s = max(alloc, key=lambda k: sizes[k] - alloc[k])
            alloc[s] += 1
        else:         # remove from the largest allocation (keep >=1)
            s = max((k for k in alloc if alloc[k] > 1), key=lambda k: alloc[k])
            alloc[s] -= 1
    return alloc
alloc = _fix_total(alloc, N_SAMPLE, sizes)
print("Stratum sizes  :", sizes)
print("Allocation     :", alloc, "-> total", sum(alloc.values()))

# --- Select within each stratum ----------------------------------------------
selected_idx = []
for s, k in alloc.items():
    sub_pos = np.where(feat[STRATUM_COL].to_numpy() == s)[0]
    X_sub = X_all[sub_pos]
    picks = select_diverse(X_sub, k, seed=SEED)
    selected_idx.extend(sub_pos[p] for p in picks)

selected_idx = sorted(set(int(i) for i in selected_idx))
sample = feat.iloc[selected_idx].copy()
print(f"\nSelected {len(sample)} workflows (method: {'k-medoids/KMeans' if _HAVE_SKLEARN else 'max-min'}).")

Stratum sizes  : {'taint': 164, 'expr-no-taint': 847, 'no-expr': 1711}
Allocation     : {'taint': 8, 'expr-no-taint': 19, 'no-expr': 27} -> total 54



Selected 54 workflows (method: k-medoids/KMeans).


In [9]:
# --- Coverage report: sample vs. population -----------------------------------
def coverage(col):
    pop = (feat[col].value_counts(normalize=True) * 100).round(1)
    smp = (sample[col].value_counts(normalize=True) * 100).round(1)
    cnt = sample[col].value_counts()
    tab = pd.concat([cnt.rename("sample_n"), smp.rename("sample_%"),
                     pop.rename("pop_%")], axis=1).fillna(0)
    return tab.sort_values("sample_n", ascending=False)

for col in ["security_surface", "trigger_risk", "complexity",
            "action_profile", "workflow_purpose", "permission_scope"]:
    print(f"\n== {col} (sample vs population) ==")
    print(coverage(col).to_string())

print("\nBinary-feature presence in the sample:")
for c in ["has_expr_in_run", "env_with_expr", "github_token", "uses_obscure",
          "has_self_hosted", "uses_container", "has_matrix", "has_dispatch_inputs"]:
    print(f"  {c:20s}: {int(sample[c].sum())}/{len(sample)}")
print(f"\n  workflows with taint_count>0 : {(sample['taint_count']>0).sum()}/{len(sample)}")
print(f"  distinct repositories        : {sample['repo'].nunique()}")


== security_surface (sample vs population) ==
                  sample_n  sample_%  pop_%
security_surface                           
no-expr                 27      50.0   62.9
expr-no-taint           19      35.2   31.1
taint                    8      14.8    6.0

== trigger_risk (sample vs population) ==
              sample_n  sample_%  pop_%
trigger_risk                           
medium              29      53.7   48.8
low                 21      38.9   39.7
high                 4       7.4   11.5

== complexity (sample vs population) ==
            sample_n  sample_%  pop_%
complexity                           
complex           25      46.3   41.5
medium            16      29.6   31.9
simple            13      24.1   26.6

== action_profile (sample vs population) ==
                sample_n  sample_%  pop_%
action_profile                           
unpinned              41      75.9   75.2
pinned                10      18.5   18.7
no-third-party         3       5.6    6.0

== 

In [ ]:
# --- Export: copy selected workflows + manifest + labeling template ----------
TOOLS = ["actionlint", "frizbee", "ggshield", "pinny", "poutine",
         "scharf", "scorecard", "semgrep", "zizmor"]

# 1) Copy the actual workflow files to accuracy/workflows_to_label/
copied, missing = 0, []
id_to_content = dict(zip(meta["workflow_id"], meta["content"]))
for _, r in sample.iterrows():
    dst = LABEL_DIR / r["src_filename"]
    src = WF_DIR / r["src_filename"]
    if src.exists():
        shutil.copy2(src, dst)
        copied += 1
    else:  # fall back to writing the stored content
        content = id_to_content.get(r["workflow_id"])
        if content:
            dst.write_text(content, encoding="utf-8")
            copied += 1
        else:
            missing.append(r["src_filename"])

print(f"Copied {copied}/{len(sample)} workflow files to {LABEL_DIR}")
if missing:
    print("MISSING:", missing)

# 2) Manifest: every metric for the chosen sample (traceability / paper table)
manifest_cols = ["workflow_id", "repo", "workflow_file", "src_filename",
                 "security_surface", "trigger_risk", "complexity", "action_profile",
                 "workflow_purpose", "permission_scope",
                 "num_jobs", "num_steps", "loc", "num_triggers",
                 "num_third_party", "num_owners", "sha_ratio", "action_run_ratio",
                 "taint_count", "secrets_count", "has_expr_in_run", "env_with_expr",
                 "github_token", "uses_obscure", "has_self_hosted", "uses_container",
                 "has_matrix", "has_needs", "uses_reusable", "has_dispatch_inputs",
                 "has_permissions", "has_conditional"]
manifest = sample[manifest_cols].sort_values(
    ["security_surface", "repo", "workflow_file"]).reset_index(drop=True)
manifest.insert(0, "sample_no", range(1, len(manifest) + 1))
manifest.to_csv(OUT_DIR / "sample_manifest.csv", index=False)

# 3) Labeling template: manual ground-truth + per-tool detection -> precision/recall
label = manifest[["sample_no", "workflow_id", "repo", "workflow_file",
                  "src_filename", "security_surface", "trigger_risk"]].copy()
label["gt_has_vulnerability"] = ""   # 1/0  (manual)
label["gt_vuln_types"] = ""          # e.g. code-injection; unpinned-action; secret-leak
label["gt_num_findings"] = ""        # optional count of true findings
label["annotator"] = ""
label["notes"] = ""
for t in TOOLS:
    label[f"det_{t}"] = ""           # 1/0 : did this tool flag the workflow?
label.to_csv(OUT_DIR / "labeling_template.csv", index=False)

print(f"\nWrote:\n  - {OUT_DIR / 'sample_manifest.csv'}")
print(f"  - {OUT_DIR / 'labeling_template.csv'}")
print(f"  - {feat_path}")
print(f"  - {LABEL_DIR}/  ({copied} workflow files)")
manifest.head(10)